# 02 Real Data Preparation

This notebook prepares the real Zindi dataset from the competition ZIP,
builds enriched train/test tables, and saves processed outputs for the
next notebooks.

In [1]:
%pip install -q pandas numpy
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from src.data.raw_competition import load_raw_competition_zip, build_current_loan_tables, build_history_features, combine_prevloans

ROOT = pathlib.Path('..').resolve()
ZIP_PATH = ROOT / 'data-science-nigeria-challenge-1-loan-default-prediction20250307-26022-im3qg9.zip'
OUT_DIR = ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

bundle = load_raw_competition_zip(str(ZIP_PATH))
train, test = build_current_loan_tables(bundle)
prev = combine_prevloans(bundle)
history = build_history_features(prev)

print(f"Train enrichi : {train.shape}")
print(f"Test enrichi  : {test.shape}")
print(f"History feats : {history.shape}")

train.to_csv(OUT_DIR / 'train_enriched.csv', index=False)
test.to_csv(OUT_DIR / 'test_enriched.csv', index=False)
history.to_csv(OUT_DIR / 'history_features.csv', index=False)
print(f"Sauvegardé dans: {OUT_DIR}")


Note: you may need to restart the kernel to use updated packages.


/workspaces/Brokerflow_ai/src/data/raw_competition.py:123: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prepared["approveddate"] = pd.to_datetime(prepared["approveddate"], errors="coerce")
/workspaces/Brokerflow_ai/src/data/raw_competition.py:124: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prepared["creationdate"] = pd.to_datetime(prepared["creationdate"], errors="coerce")


Train enrichi : (4368, 26)
Test enrichi  : (1450, 24)
History feats : (5801, 8)
Sauvegardé dans: /workspaces/Brokerflow_ai/data/processed


In [2]:
import pandas as pd

print("=== Contrôle rapide des données réelles ===\n")
print("target_risk_flag (train) :")
print(train['target_risk_flag'].value_counts(dropna=False).rename({0: 'Good (0)', 1: 'Bad (1)'}))

print("\nTaux de défaut global :")
print(f"{train['target_risk_flag'].mean():.2%}")

key_cols = ['customerid', 'loanid', 'approveddate', 'birthdate', 'bank_account_type']
existing = [c for c in key_cols if c in train.columns]
print("\nAperçu colonnes clés :")
print(train[existing].head(5))

print("\nValeurs manquantes (top 10) :")
print(train.isna().sum().sort_values(ascending=False).head(10))


=== Contrôle rapide des données réelles ===

target_risk_flag (train) :
target_risk_flag
Good (0)    3416
Bad (1)      952
Name: count, dtype: int64

Taux de défaut global :
21.79%

Aperçu colonnes clés :
                         customerid        approveddate  birthdate  \
0  8a2a81a74ce8c05d014cfb32a0da1049 2017-07-25 08:22:56 1972-01-15   
1  8a85886e54beabf90154c0a29ae757c0 2017-07-05 17:04:41 1985-08-23   
2  8a8588f35438fe12015444567666018e 2017-07-06 14:52:57 1984-09-18   
3  8a85890754145ace015429211b513e16 2017-07-27 19:00:41 1977-10-10   
4  8a858970548359cc0154883481981866 2017-07-03 23:42:45 1986-09-07   

  bank_account_type  
0             Other  
1           Savings  
2             Other  
3           Savings  
4             Other  

Valeurs manquantes (top 10) :
bank_branch_clients                4325
referredby                         3781
level_of_education_clients         3763
employment_status_clients           657
ever_late_flag                        9
max_late_da